In [6]:
import os
import pandas as pd

from langchain_ollama import ChatOllama
from langchain_anthropic import ChatAnthropic
from src.graphs.IdentificationGraph import build_identification_graph
from datetime import datetime


In [9]:
MAX_TRIALS = 10
# model = ChatOllama(
#     model="nemotron-3-ultra:cloud",
#     temperature=0,
#     reasoning=False
# )

model = ChatAnthropic(
    model="claude-sonnet-4-6",
    temperature=0,
    max_tokens=4096,
    api_key=""
)



In [10]:
# 1. Define the Graph
descriptions = {
    "lib_desc": open("../inputs/descriptions/library.txt", encoding="utf-8").read(),
    "rental_desc": open("../inputs/descriptions/car_rental.txt", encoding="utf-8").read(),
    "ntss_desc": open("../inputs/descriptions/ntss.txt", encoding="utf-8").read(),
}

# 2. Load ground truths (upload library.csv, car_rental.csv, ntss.csv to
#    ground_truths/identification/ first - see that folder's README for the format)
try:
    truths = {
        "lib_truth": pd.read_csv("../ground_truths/identification/lib.csv"),
        "rent_truth": pd.read_csv("../ground_truths/identification/rental.csv"),
        "ntss_truth": pd.read_csv("../ground_truths/identification/ntss.csv"),
    }
except FileNotFoundError as e:
    raise FileNotFoundError(
        "Identification ground truths not found. Upload library.csv, car_rental.csv and "
        "ntss.csv to ground_truths/identification/ (columns: rule,phrase)"
    ) from e

workflow = build_identification_graph(
    model=model,
    descriptions=descriptions,
    truths=truths,
    max_trials=MAX_TRIALS
)
agent = workflow.compile()

# 3. Run the Optimizer
# Pulling initial prompt from the specified file
prompt_path = "../prompts/auto_prompt_evo/identification/t0/prompt.md"
try:
    with open(prompt_path, "r", encoding="utf-8") as f:
        initial_prompt = f.read()
    print("Successfully loaded prompt from:", prompt_path)
except FileNotFoundError:
    raise FileNotFoundError(f"Could not find file at {prompt_path}. Please check the path.")

if "<DESCRIPTION>" not in initial_prompt:
    raise ValueError("Initial prompt is missing the <DESCRIPTION> placeholder - "
                     "descriptions would never be injected!")

inputs = {
    "current_prompt": initial_prompt,
    "current_trial": 1,
    "lib_eval": {}, "rental_eval": {}, "ntss_eval": {}
}

print(f"\n[{datetime.now():%H:%M:%S}] Starting identification optimizer: {MAX_TRIALS} trials planned")
result = agent.invoke(inputs)
print(f"\n[{datetime.now():%H:%M:%S}] Optimizer finished after {result['current_trial'] - 1} completed trials")


Successfully loaded prompt from: ../prompts/auto_prompt_evo/identification/t0/prompt.md

[01:50:21] Starting identification optimizer: 10 trials planned
[01:50:21] Trial 1 | lib: identifying domain phrases...
[01:50:29] Trial 1 | lib: done (1032 chars)
[01:50:29] Trial 1 | rental: identifying domain phrases...
[01:50:38] Trial 1 | rental: done (1297 chars)
[01:50:38] Trial 1 | ntss: identifying domain phrases...
[01:50:46] Trial 1 | ntss: done (1392 chars)

--- Identification Evaluation Results (trial 1) ---
Lib: p-0.63 r-0.62 f1-0.62 (TP=34 FP=20 FN=21)
Rent: p-0.46 r-0.60 f1-0.52 (TP=37 FP=44 FN=25)
Ntss: p-0.67 r-0.66 f1-0.66 (TP=57 FP=28 FN=30)
[01:50:46] Trial 1 | optimize: asking model to improve the prompt...
[01:51:28] Trial 1 | optimize: done
[01:51:28] Trial 1 | checkpoint: saving optimized prompt to ../prompts/auto_prompt_evo/identification/t1/prompt.md
[01:51:28] Trial 2 | lib: identifying domain phrases...
[01:51:34] Trial 2 | lib: done (934 chars)
[01:51:34] Trial 2 | ren

KeyboardInterrupt: 